# Day 5 — MPS, Truncation, and the Heisenberg Chain

**1. Matrix Product State (MPS)**

A normalized four-qubit state is decomposed using sequential SVD into four tensors, (A_1,A_2,A_3,A_4). Contracting these tensors reconstructs the original state with negligible numerical error.

**2. Singular Values and Entanglement**

For a sample coefficient matrix, singular values are squared to obtain probabilities, which are used to calculate entanglement entropy.

**3. Bond Dimension and Truncation**

The bond dimension controls the size of MPS tensors. Discarding smaller singular values reduces computational cost but introduces approximation error. The truncation examples illustrate this trade-off.

**4. Heisenberg Chain**

A three-spin Heisenberg Hamiltonian is constructed using Pauli matrices. Its ground state is obtained through diagonalization, then reshaped into a tensor. SVD gives the Schmidt coefficients and entanglement entropy across the (1|23) bipartition.

**Key idea:** SVD connects state decomposition, tensor size, truncation error, and entanglement analysis.

## Sequential SVD: Building an MPS for a 4-Site Random State

In [ ]:
import numpy as np

n_sites = 4
psi = np.random.randn(2**n_sites) + 1j * np.random.randn(2**n_sites)

psi = psi / np.linalg.norm(psi)

print("Shape:", psi.shape)
print("Norm:", np.linalg.norm(psi))

Shape: (16,)
Norm: 1.0


In [ ]:
psi_tensor = psi.reshape(2, 2, 2, 2)

print("Shape:", psi_tensor.shape)

Shape: (2, 2, 2, 2)


In [ ]:
M = psi_tensor.reshape(2, 8)

print("Shape:", M.shape)

Shape: (2, 8)


In [ ]:
U, S, Vh = np.linalg.svd(M, full_matrices=False)

print("U shape:", U.shape)
print("S shape:", S.shape)
print("Vh shape:", Vh.shape)
print("Singular values:", S)

U shape: (2, 2)
S shape: (2,)
Vh shape: (2, 8)
Singular values: [0.84528364 0.53431785]


In [ ]:
D1 = len(S)

A1 = U.reshape(1, 2, D1)

print("A1 shape:", A1.shape)

A1 shape: (1, 2, 2)


In [ ]:
R1 = np.diag(S) @ Vh

print("R1 shape:", R1.shape)

R1 shape: (2, 8)


In [ ]:
M2 = R1.reshape(D1 * 2, 4)

print("M2 shape:", M2.shape)

M2 shape: (4, 4)


In [ ]:
U2, S2, Vh2 = np.linalg.svd(M2, full_matrices=False)

print("U2 shape:", U2.shape)
print("S2 shape:", S2.shape)
print("Vh2 shape:", Vh2.shape)
print("S2:", S2)

U2 shape: (4, 4)
S2 shape: (4,)
Vh2 shape: (4, 4)
S2: [0.79220073 0.47391428 0.36678908 0.11527806]


In [ ]:
R2 = np.diag(S2) @ Vh2

print("R2 shape:", R2.shape)

R2 shape: (4, 4)


In [ ]:
D2 = len(S2)

A2 = U2.reshape(D1, 2, D2)

print("D2:", D2)
print("A2 shape:", A2.shape)

D2: 4
A2 shape: (2, 2, 4)


In [ ]:
M3 = R2.reshape(D2 * 2, 2)

print("M3 shape:", M3.shape)

M3 shape: (8, 2)


In [ ]:
U3, S3, Vh3 = np.linalg.svd(M3, full_matrices=False)

print("U3 shape:", U3.shape)
print("S3 shape:", S3.shape)
print("Vh3 shape:", Vh3.shape)
print("S3:", S3)

U3 shape: (8, 2)
S3 shape: (2,)
Vh3 shape: (2, 2)
S3: [0.85908716 0.51182931]


In [ ]:
R3 = np.diag(S3) @ Vh3

print("R3 shape:", R3.shape)

R3 shape: (2, 2)


In [ ]:
D3 = len(S3)

A3 = U3.reshape(D2, 2, D3)

print("D3:", D3)
print("A3 shape:", A3.shape)

D3: 2
A3 shape: (4, 2, 2)


In [ ]:
A4 = R3.reshape(D3, 2, 1)

print("A4 shape:", A4.shape)

A4 shape: (2, 2, 1)


In [ ]:
M12 = np.tensordot(A1, A2, axes=([2], [0]))

print("M12 shape:", M12.shape)

M12 shape: (1, 2, 2, 4)


In [ ]:
M123 = np.tensordot(M12, A3, axes=([3], [0]))

print("M123 shape:", M123.shape)

M123 shape: (1, 2, 2, 2, 2)


In [ ]:
M1234 = np.tensordot(M123, A4, axes=([4], [0]))

print("M1234 shape:", M1234.shape)

M1234 shape: (1, 2, 2, 2, 2, 1)


In [ ]:
psi_reconstructed = np.squeeze(M1234)

print("Shape:", psi_reconstructed.shape)

Shape: (2, 2, 2, 2)


In [ ]:
psi_reconstructed = psi_reconstructed.reshape(-1)

print("Shape:", psi_reconstructed.shape)

Shape: (16,)


In [ ]:
error = np.linalg.norm(psi_reconstructed - psi)

print("Reconstruction error:", error)

Reconstruction error: 1.0194541270147583e-15


## Schmidt Coefficients and Entanglement Entropy

In [ ]:
C = np.array([
[0.8, 0.0],
[0.0, 0.6] ])

U, S, Vh = np.linalg.svd(C)

check = U.conj().T @ U
print("U:")
print(U)

print("\nU†U:")
print(check)

U:
[[1. 0.]
 [0. 1.]]

U†U:
[[1. 0.]
 [0. 1.]]


In [ ]:
print("Singular values:", S)

p = S**2

print("Probabilities:", p)
print("Sum:", np.sum(p))

Singular values: [0.8 0.6]
Probabilities: [0.64 0.36]
Sum: 1.0


In [ ]:
entropy = -np.sum(p * np.log2(p))

print("Entanglement entropy:", entropy)

Entanglement entropy: 0.9426831892554921


In [ ]:
A1_matrix = A1.reshape(2, 2)

check = A1_matrix.conj().T @ A1_matrix

print("A1† A1:")
print(check)

A1† A1:
[[1.00000000e+00+0.00000000e+00j 5.55111512e-17+1.38777878e-17j]
 [5.55111512e-17-1.38777878e-17j 1.00000000e+00+0.00000000e+00j]]


In [ ]:
S = np.array([0.80, 0.50, 0.10, 0.02])

Dmax = 2

S_kept = S[:Dmax]
S_discarded = S[Dmax:]

print("Kept:", S_kept)
print("Discarded:", S_discarded)

Kept: [0.8 0.5]
Discarded: [0.1  0.02]


In [ ]:
error_squared = np.sum(S_discarded**2)
error = np.sqrt(error_squared)

print("Squared truncation error:", error_squared)
print("Truncation error:", error)

Squared truncation error: 0.010400000000000001
Truncation error: 0.10198039027185571


In [ ]:
D = 2
elements = 2 * D**2
print("D =", D, "elements =", elements)

D = 10
elements = 2 * D**2
print("D =", D, "elements =", elements)

D = 50
elements = 2 * D**2
print("D =", D, "elements =", elements)

D = 100
elements = 2 * D**2
print("D =", D, "elements =", elements)

D = 2 elements = 8
D = 10 elements = 200
D = 50 elements = 5000
D = 100 elements = 20000


## Truncation and Bond Dimension Cutoff

In [ ]:
S = np.array([0.80, 0.50, 0.10, 0.02])

# Dmax = 2
discarded_2 = S[2:]
error_squared_2 = np.sum(discarded_2**2)
error_2 = np.sqrt(error_squared_2)

# Dmax = 3
discarded_3 = S[3:]
error_squared_3 = np.sum(discarded_3**2)
error_3 = np.sqrt(error_squared_3)

print("Dmax = 2, Error =", error_2)
print("Dmax = 3, Error =", error_3)

Dmax = 2, Error = 0.10198039027185571
Dmax = 3, Error = 0.02


In [ ]:
d = 2

D = 10
size_10 = D * d * D

D = 50
size_50 = D * d * D

print("D=10:", size_10)
print("D=50:", size_50)

D=10: 200
D=50: 5000


## The Heisenberg Hamiltonian (3-Site Chain)

In [ ]:
I = np.eye(2, dtype=complex)

X = np.array([
[0, 1],
[1, 0]
], dtype=complex)

Y = np.array([
[0, -1j],
[1j, 0]
], dtype=complex)

Z = np.array([
[1, 0],
[0, -1]
], dtype=complex)

print("I:\n", I)
print("X:\n", X)
print("Y:\n", Y)
print("Z:\n", Z)

I:
 [[1.+0.j 0.+0.j]
 [0.+0.j 1.+0.j]]
X:
 [[0.+0.j 1.+0.j]
 [1.+0.j 0.+0.j]]
Y:
 [[ 0.+0.j -0.-1.j]
 [ 0.+1.j  0.+0.j]]
Z:
 [[ 1.+0.j  0.+0.j]
 [ 0.+0.j -1.+0.j]]


In [ ]:
X1X2 = np.kron(np.kron(X, X), I)

print("Shape:", X1X2.shape)
print(X1X2)

Shape: (8, 8)
[[0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 1.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 1.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 1.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]]


In [ ]:
Y1Y2 = np.kron(np.kron(Y, Y), I)
Z1Z2 = np.kron(np.kron(Z, Z), I)

print("Y1Y2 shape:", Y1Y2.shape)
print("Z1Z2 shape:", Z1Z2.shape)

Y1Y2 shape: (8, 8)
Z1Z2 shape: (8, 8)


In [ ]:
H12 = X1X2 + Y1Y2 + Z1Z2

print("H12 shape:", H12.shape)

H12 shape: (8, 8)


In [ ]:
X2X3 = np.kron(I, np.kron(X, X))
Y2Y3 = np.kron(I, np.kron(Y, Y))
Z2Z3 = np.kron(I, np.kron(Z, Z))

print("X2X3 shape:", X2X3.shape)
print("Y2Y3 shape:", Y2Y3.shape)
print("Z2Z3 shape:", Z2Z3.shape)

X2X3 shape: (8, 8)
Y2Y3 shape: (8, 8)
Z2Z3 shape: (8, 8)


In [ ]:
H23 = X2X3 + Y2Y3 + Z2Z3

print("H23 shape:", H23.shape)

H23 shape: (8, 8)


In [ ]:
H = H12 + H23

print("Hamiltonian shape:", H.shape)

Hamiltonian shape: (8, 8)


In [ ]:
hermitian_check = np.allclose(H, H.conj().T)

print("Hermitian:", hermitian_check)

Hermitian: True


In [ ]:
eigenvalues, eigenvectors = np.linalg.eigh(H)

print("Energies:")
print(eigenvalues)

Energies:
[-4.00000000e+00 -4.00000000e+00 -2.17382536e-16 -2.17382536e-16
  2.00000000e+00  2.00000000e+00  2.00000000e+00  2.00000000e+00]


In [ ]:
ground_energy = eigenvalues[0]
ground_state = eigenvectors[:, 0]

print("Ground energy:", ground_energy)
print("Ground-state shape:", ground_state.shape)

Ground energy: -4.0
Ground-state shape: (8,)


In [ ]:
norm = np.vdot(ground_state, ground_state)

print("Norm:", norm)

Norm: (0.9999999999999998+0j)


In [ ]:
residual = H @ ground_state - ground_energy * ground_state

print("Residual norm:", np.linalg.norm(residual))

Residual norm: 1.0175362097255202e-15


## Ground State Entanglement Entropy

In [ ]:
ground_tensor = ground_state.reshape(2, 2, 2)

print("Ground tensor shape:", ground_tensor.shape)

Ground tensor shape: (2, 2, 2)


In [ ]:
M = ground_tensor.reshape(2, 4)

print("Matrix shape:", M.shape)

Matrix shape: (2, 4)


In [ ]:
U, S, Vh = np.linalg.svd(M, full_matrices=False)

print("U shape:", U.shape)
print("S:", S)
print("Vh shape:", Vh.shape)

U shape: (2, 2)
S: [0.91287093 0.40824829]
Vh shape: (2, 4)


In [ ]:
probabilities = S**2

print("Probabilities:", probabilities)
print("Sum:", np.sum(probabilities))

Probabilities: [0.83333333 0.16666667]
Sum: 0.9999999999999998


In [ ]:
entropy = -np.sum(probabilities * np.log2(probabilities))

print("Entanglement entropy:", entropy)

Entanglement entropy: 0.6500224216483548
